# B^± → π^± π^+ π^-: CPV isobar self-closure

This is the CPV baseline closure before returning to QMI.

- generator and fit use the **same isobaric model**;
- toy generation uses **accept-reject**, avoiding the narrow-ω bias seen with the 1024 inverse-transform grid;
- normalization uses `gauss-legendre`; the narrow ω(782) bands are handled automatically by the adaptive normalization;
- there is no efficiency, background, veto, or detector smearing;
- the ρ(770) fixes the two phase conventions with x=1, y=0 and dy=0;
- rho770.dx is floated, with a bound selecting the physical small-|dx| branch.

A successful closure here isolates the CP machinery from the QMI parametrization.


In [ ]:
import numpy as np
import pandas as pd
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dataclasses import dataclass
from dalitzplotfitter import (
    CPFitSession, CPRealImag, DecayChannel, DecayModel,
    GounarisSakurai, Parameter, RelativisticBreitWigner,
    Resonance, enable_x64, generate_cp_toy,
)

enable_x64()

SEED = 20260905
N_EVENTS = 250_000
NORMALIZATION_METHOD = "gauss-legendre"
NORMALIZATION_BIN_WIDTH = 0.005
TOY_METHOD = "accept-reject"

channel_plus = DecayChannel("B+", ("pi+", "pi+", "pi-"))
channel_minus = DecayChannel("B-", ("pi-", "pi-", "pi+"))
M_PI = float(channel_plus.daughter_masses[0])

TRUTH_COEFFICIENTS = {
    "rho770":       dict(x= 1.000, y= 0.000, dx=-0.003, dy= 0.000),
    "omega782":     dict(x= 0.091, y=-0.007, dx= 0.000, dy=-0.022),
    "f2_1270":      dict(x= 0.291, y= 0.204, dx=-0.002, dy=-0.179),
    "rho1450":      dict(x=-0.223, y= 0.191, dx= 0.031, dy= 0.068),
    "rho3_1690":    dict(x= 0.073, y=-0.045, dx= 0.044, dy=-0.013),
    "sigma":        dict(x=-0.485, y= 0.284, dx= 0.231, dy= 0.270),
}

def truth_cp(name, charge):
    p = TRUTH_COEFFICIENTS[name]
    return CPRealImag(p["x"], p["y"], p["dx"], p["dy"], charge=charge)

def coefficient_acp(values):
    x, y, dx, dy = (float(values[k]) for k in ("x", "y", "dx", "dy"))
    return -2.0*(x*dx + y*dy)/(x*x + y*y + dx*dx + dy*dy)

print("toy size:", N_EVENTS)
print("normalization:", NORMALIZATION_METHOD, "bin width =", NORMALIZATION_BIN_WIDTH, "GeV")


In [ ]:
@dataclass(frozen=True)
class PaperSigmaPole:
    def __call__(self, mass, context):
        m = jnp.asarray(mass)
        pole = jnp.asarray(context.pole_mass) - 1j*jnp.asarray(context.pole_width)
        return 1.0/(pole**2 - m**2)

def isobar_components(coefficient_for):
    return [
        Resonance("rho770", (0,2), coefficient_for("rho770"),
                  mass=0.7708, width=0.1534, spin=1,
                  lineshape=GounarisSakurai(),
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("omega782", (0,2), coefficient_for("omega782"),
                  mass=0.78265, width=0.00849, spin=1,
                  lineshape=RelativisticBreitWigner(),
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("f2_1270", (0,2), coefficient_for("f2_1270"),
                  mass=1.2755, width=0.1867, spin=2,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho1450", (0,2), coefficient_for("rho1450"),
                  mass=1.465, width=0.400, spin=1,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho3_1690", (0,2), coefficient_for("rho3_1690"),
                  mass=1.6888, width=0.161, spin=3,
                  resonance_radius=4.0, parent_radius=4.0),
        Resonance("sigma", (0,2), coefficient_for("sigma"),
                  mass=0.563, width=0.350, spin=0,
                  lineshape=PaperSigmaPole(),
                  resonance_radius=4.0, parent_radius=4.0),
    ]

generator_plus = DecayModel(
    channel_plus,
    isobar_components(lambda name: truth_cp(name,+1)),
    normalize_components=True,
    normalization_method=NORMALIZATION_METHOD,
    normalization_bin_width=NORMALIZATION_BIN_WIDTH,
)
generator_minus = DecayModel(
    channel_minus,
    isobar_components(lambda name: truth_cp(name,-1)),
    normalize_components=True,
    normalization_method=NORMALIZATION_METHOD,
    normalization_bin_width=NORMALIZATION_BIN_WIDTH,
)

print("B+ normalization scheme:", generator_plus.normalization_scheme)
print("B- normalization scheme:", generator_minus.normalization_scheme)


In [ ]:
toy_plus, toy_minus = generate_cp_toy(
    generator_plus, generator_minus, N_EVENTS,
    seed=SEED,
    method=TOY_METHOD,
    pool_size=1_000_000,
    envelope_safety=1.5,
    max_restarts=20,
)

n_total = toy_plus.size + toy_minus.size
print("B+ events:", toy_plus.size)
print("B- events:", toy_minus.size)
print("raw charge asymmetry =", (toy_minus.size-toy_plus.size)/n_total)

def low_pipi_mass(sample):
    return np.sqrt(np.minimum(np.asarray(sample.s13), np.asarray(sample.s23)))

fig, axes = plt.subplots(2,2,figsize=(12,10),constrained_layout=True)
for ax, sample, label in [
    (axes[0,0],toy_plus,"B+"),
    (axes[0,1],toy_minus,"B-"),
]:
    h=ax.hist2d(np.asarray(sample.s13),np.asarray(sample.s23),bins=120)
    ax.set_title(label+" toy Dalitz")
    ax.set_xlabel(r"$s_{13}$ [GeV$^2$]")
    ax.set_ylabel(r"$s_{23}$ [GeV$^2$]")
    fig.colorbar(h[3],ax=ax)

bins=np.linspace(2*M_PI,3.0,90)
axes[1,0].hist(low_pipi_mass(toy_plus),bins=bins,histtype="step",label="B+")
axes[1,0].hist(low_pipi_mass(toy_minus),bins=bins,histtype="step",label="B-")
axes[1,0].set_xlabel(r"$m_{low}(\pi^+\pi^-)$ [GeV]")
axes[1,0].legend()

smin=min(np.min(np.asarray(toy_plus.s13)),np.min(np.asarray(toy_minus.s13)))
smax=max(np.max(np.asarray(toy_plus.s13)),np.max(np.asarray(toy_minus.s13)))
sbins=np.linspace(smin,smax,90)
axes[1,1].hist(np.asarray(toy_plus.s13),bins=sbins,histtype="step",label="B+")
axes[1,1].hist(np.asarray(toy_minus.s13),bins=sbins,histtype="step",label="B-")
axes[1,1].set_xlabel(r"$s_{13}$ [GeV$^2$]")
axes[1,1].legend()
plt.show()


## Identical CPV fit model

For the rho770 reference, x, y and dy are fixed. The parameter dx remains free so that the reference component can carry direct CP violation. For closure we use `bounds=(-0.5,0.5)` on rho770.dx to remove the large-|dx| branch of the reference asymmetry.


In [ ]:
def make_fit_cp(name):
    t=TRUTH_COEFFICIENTS[name]
    reference=name=="rho770"
    return CPRealImag(
        Parameter.coefficient(f"{name}.x",t["x"],owner=name,
                              fixed=reference,step=0.005),
        Parameter.coefficient(f"{name}.y",t["y"],owner=name,
                              fixed=reference,step=0.005),
        Parameter.coefficient(f"{name}.dx",t["dx"],owner=name,
                              fixed=False,
                              bounds=(-0.5,0.5) if reference else None,
                              step=0.005),
        Parameter.coefficient(f"{name}.dy",t["dy"],owner=name,
                              fixed=reference,step=0.005),
    )

FIT_CP={name:make_fit_cp(name) for name in TRUTH_COEFFICIENTS}

fit_plus=DecayModel(
    channel_plus,
    isobar_components(lambda name: FIT_CP[name].for_charge(+1)),
    normalize_components=True,
    normalization_method=NORMALIZATION_METHOD,
    normalization_bin_width=NORMALIZATION_BIN_WIDTH,
)
fit_minus=DecayModel(
    channel_minus,
    isobar_components(lambda name: FIT_CP[name].for_charge(-1)),
    normalize_components=True,
    normalization_method=NORMALIZATION_METHOD,
    normalization_bin_width=NORMALIZATION_BIN_WIDTH,
)

session=CPFitSession(fit_plus,fit_minus,toy_plus,toy_minus)

print("parameters:",len(session.parameters))
print("free:",sum(not p.fixed for p in session.parameters))
for p in session.parameters:
    print(f"{p.name:20s} start={float(p.value): .6f} fixed={p.fixed} bounds={p.bounds}")


In [ ]:
starts={p.name:float(p.value) for p in session.parameters if not p.fixed}
nll_start=float(session.objective(starts))

gradient_check=session.minimizer(
    tolerance=1e-4,verbose=0
).check_gradient(
    starts,step_scale=1e-5,print_table=False
)

print("NLL(start) =",nll_start)
print("max |JAX-finite diff gradient error| =",gradient_check.max_absolute_error)


In [ ]:
result=session.fit(
    start_values=starts,
    simplex=False,
    strategy=2,
    hesse=True,
    tolerance=1e-4,
    verbose=1,
)
fit_values=session.result_values(result)

print("valid =",bool(result.valid))
print("NLL(start) =",nll_start)
print("NLL(fit)   =",float(result.fval))
print("Delta NLL  =",nll_start-float(result.fval))
print("EDM        =",float(result.fmin.edm))
print("nfcn       =",int(result.nfcn))


In [ ]:
rows=[]
for name,truth in TRUTH_COEFFICIENTS.items():
    for field in ("x","y","dx","dy"):
        pname=f"{name}.{field}"
        par=next(p for p in session.parameters if p.name==pname)
        fitted=fit_values[pname]
        error=0.0 if par.fixed else float(result.errors[pname])
        pull=np.nan if error==0.0 else (fitted-truth[field])/error
        rows.append(dict(
            component=name,parameter=field,truth=truth[field],
            fit=fitted,error=error,pull=pull,fixed=par.fixed,
        ))

cp_table=pd.DataFrame(rows)
display(cp_table)

free_pulls=cp_table.loc[~cp_table["fixed"],"pull"].to_numpy()
print("max |pull| =",np.nanmax(np.abs(free_pulls)))
print("RMS pull   =",np.sqrt(np.nanmean(free_pulls**2)))

acp_rows=[]
for name,truth in TRUTH_COEFFICIENTS.items():
    fitted={field:fit_values[f"{name}.{field}"] for field in ("x","y","dx","dy")}
    acp_rows.append(dict(
        component=name,
        Acp_truth=coefficient_acp(truth),
        Acp_fit=coefficient_acp(fitted),
        difference=coefficient_acp(fitted)-coefficient_acp(truth),
    ))
display(pd.DataFrame(acp_rows))


In [ ]:
session.print_fit_fractions(
    result,
    acceptance_weighted=False,
    include_interference=False,
    precision=4,
)

for variable in ("s13","s23"):
    session.plot_projection(
        result,variable,
        bins=80,
        show_components=False,
        projection_size=500_000,
        projection_seed=SEED+100,
    )
    plt.show()


## Closure criteria

The CPV isobaric self-closure should show:

- a valid simultaneous fit with small EDM;
- a small JAX vs finite-difference gradient discrepancy at the truth point;
- Cartesian CP-parameter pulls compatible with zero;
- good B+ and B- projections;
- coefficient-level CP asymmetries compatible with the generated values.

If this closes, the next controlled step is to replace only the isobaric S-wave by QMI while keeping this exact CP and normalization setup.
